<a href="https://colab.research.google.com/github/shivanvithajayam/1BF24CS127_AI_LAB/blob/main/Unification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import re

def parse_expression(expr_str):
    expr_str = expr_str.replace(" ", "")
    tokens = re.split(r'([(),])', expr_str)
    tokens = [t for t in tokens if t]

    def parse_tokens(token_list):
        if not token_list:
            return None

        token = token_list.pop(0)

        if token_list and token_list[0] == '(':
            token_list.pop(0)
            args = []
            while token_list and token_list[0] != ')':
                arg = parse_tokens(token_list)
                if arg:
                    args.append(arg)
                if token_list and token_list[0] == ',':
                    token_list.pop(0)
            if token_list:
                token_list.pop(0)
            return [token] + args
        else:
            return token

    return parse_tokens(tokens)

def is_variable(x):
    return isinstance(x, str) and x.islower() and len(x) == 1

def is_constant(x):
    return isinstance(x, str) and x.isupper() and len(x) == 1

def is_function(x):
    return isinstance(x, list) and isinstance(x[0], str) and x[0].islower()

def occurs_in(var, expr):
    if var == expr:
        return True
    if isinstance(expr, list):
        return any(occurs_in(var, sub_expr) for sub_expr in expr[1:])
    return False

def apply_substitution(subst, expr):
    if not subst:
        return expr
    if isinstance(expr, str):
        return subst.get(expr, expr)
    if isinstance(expr, list):
        return [expr[0]] + [apply_substitution(subst, sub) for sub in expr[1:]]
    return expr

def format_expression(expr):
    if isinstance(expr, str):
        return expr
    if isinstance(expr, list):
        return f"{expr[0]}({', '.join(format_expression(arg) for arg in expr[1:])})"
    return str(expr)

def unify(psi1, psi2):
    if isinstance(psi1, str) or isinstance(psi2, str):
        if psi1 == psi2:
            return {}
        elif is_variable(psi1):
            if occurs_in(psi1, psi2):
                return "FAILURE"
            else:
                return {psi1: psi2}
        elif is_variable(psi2):
            if occurs_in(psi2, psi1):
                return "FAILURE"
            else:
                return {psi2: psi1}
        else:
            return "FAILURE"

    if psi1[0] != psi2[0]:
        return "FAILURE"

    if len(psi1) != len(psi2):
        return "FAILURE"

    subst = {}
    args1 = psi1[1:]
    args2 = psi2[1:]

    for i in range(len(args1)):
        s = unify(args1[i], args2[i])

        if s == "FAILURE":
            return "FAILURE"

        if s != {}:
            args1 = [apply_substitution(s, arg) for arg in args1]
            args2 = [apply_substitution(s, arg) for arg in args2]

            for var in subst:
                subst[var] = apply_substitution(s, subst[var])
            subst.update(s)

    return subst

if __name__ == "__main__":
    print("--- First-Order Logic Unification System ---")
    print("Rules: Uppercase (A, B) = Constants | Lowercase (x, y) = Variables")
    print("Examples: 'P(A, B, B)' and 'P(x, y, z)' OR 'P(x, f(x))' and 'P(A, f(A))'\n")

    input1 = input("Enter Expression 1 (Ψ1): ").strip()
    input2 = input("Enter Expression 2 (Ψ2): ").strip()

    try:
        expr1 = parse_expression(input1)
        expr2 = parse_expression(input2)

        if not expr1 or not expr2 or not isinstance(expr1, list) or not isinstance(expr2, list):
            raise ValueError("Invalid format. Make sure to use standard predicate form like P(...)")

        result = unify(expr1, expr2)

        print("\n--- Results ---")
        print(f"Parsed Ψ1: {format_expression(expr1)}")
        print(f"Parsed Ψ2: {format_expression(expr2)}")
        print("Substitution Set (SUBST):")

        if result == "FAILURE":
            print("Unification Failed (Mismatched symbols or Occur Check failed).")
        elif result == {}:
            print("NIL (Expressions are already structurally identical)")
        else:
            formatted_subst = ", ".join([f"({format_expression(val)}/{key})" for key, val in result.items()])
            print(f"{{ {formatted_subst} }}")

    except Exception as e:
        print(f"\nParsing Error: Please check your syntax format. ({e})")


--- First-Order Logic Unification System ---
Rules: Uppercase (A, B) = Constants | Lowercase (x, y) = Variables
Examples: 'P(A, B, B)' and 'P(x, y, z)' OR 'P(x, f(x))' and 'P(A, f(A))'

Enter Expression 1 (Ψ1): Q(a,g(x,a),f(y))
Enter Expression 2 (Ψ2): Q(a,g(f(b),a),x)

--- Results ---
Parsed Ψ1: Q(a, g(x, a), f(y))
Parsed Ψ2: Q(a, g(f(b), a), x)
Substitution Set (SUBST):
{ (f(b)/x), (b/y) }
